# Modele 3 - XGBoost
Gradient boosting performant (pas besoin de scaling).
Etapes: train/test, evaluation, cross-validation, et tuning optionnel.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

In [ ]:
def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data' / 'customer_churn.csv').exists():
            return candidate
    raise FileNotFoundError('Impossible de localiser data/customer_churn.csv')

PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / 'data' / 'customer_churn.csv'

df = pd.read_csv(DATA_PATH)
if 'customer_id' in df.columns:
    df = df.drop(columns=['customer_id'])

target_col = 'churn'
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
def apply_feature_engineering(data):
    temp = data.copy()
    if 'tenure_months' in temp.columns:
        tenure_safe = temp['tenure_months'].replace(0, np.nan)
    else:
        tenure_safe = None

    if {'support_tickets', 'tenure_months'}.issubset(temp.columns):
        temp['tickets_per_tenure'] = (temp['support_tickets'] / tenure_safe).fillna(0)
    if {'total_revenue', 'tenure_months'}.issubset(temp.columns):
        temp['revenue_per_month'] = (temp['total_revenue'] / tenure_safe).fillna(0)
    if {'payment_failures', 'tenure_months'}.issubset(temp.columns):
        temp['payment_failure_rate'] = (temp['payment_failures'] / tenure_safe).fillna(0)
    return temp

X_train = apply_feature_engineering(X_train)
X_test = apply_feature_engineering(X_test)

num_cols = X_train.select_dtypes(include=['number', 'bool']).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=['number', 'bool']).columns.tolist()

num_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

cat_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

In [ ]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
pipeline = ImbPipeline([
    ('pre', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('model', model)
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, values_format='d')
plt.title('XGBoost - Confusion Matrix')
plt.show()

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {'recall': 'recall', 'f1': 'f1', 'roc_auc': 'roc_auc'}
cv_results = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
cv_summary = {metric: cv_results[f'test_{metric}'].mean() for metric in scoring}
print('CV mean scores:', cv_summary)

## Tuning optionnel (RandomizedSearchCV)
A activer uniquement si ce modele fait partie des 2 meilleurs.

In [ ]:
run_search = False
if run_search:
    param_distributions = {
        'model__n_estimators': [200, 400, 600],
        'model__max_depth': [3, 5, 7],
        'model__learning_rate': [0.05, 0.1, 0.2],
        'model__subsample': [0.8, 1.0],
        'model__colsample_bytree': [0.8, 1.0]
    }
    search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_distributions,
        n_iter=10,
        cv=cv,
        scoring='recall',
        n_jobs=-1,
        random_state=42
    )
    search.fit(X_train, y_train)
    print('Best params:', search.best_params_)
    y_pred_best = search.best_estimator_.predict(X_test)
    print(classification_report(y_test, y_pred_best))